# Pandas: A Refresher Lesson

Pandas is the main library for working with tabular data, meaning data in rows and columns, like a spreadsheet or a database table. It's built on NumPy arrays.

This lesson goes in order:

1. Series and DataFrame basics
2. Loading data
3. Inspecting data
4. Selecting columns and rows (`[]`, `loc`, `iloc`)
5. Filtering
6. Creating and changing columns
7. Missing values
8. Sorting and ranking
9. Grouping and aggregating (`groupby`)
10. Combining tables (`merge` and `concat`)
11. Dates
12. Chaining operations
13. Practice exercises, with solutions

We'll use a small, made-up dataset of health insurance claims, so the code runs without any downloads. Run the cells in order.

## 1. Setup and core objects

Pandas has two main objects:

- **`Series`**: one column of values, with an index (row labels).
- **`DataFrame`**: a table made of several Series that share the same index.

In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)

s = pd.Series([120, 45, 890], name="claim_cost")
print(s)

A DataFrame is a dictionary of columns, or a list of row dictionaries:

In [ ]:
df = pd.DataFrame({
    "name": ["Ana", "Ben", "Chloe"],
    "age": [34, 58, 41],
    "claim_cost": [120.0, 890.0, 45.0],
})
df

In a notebook, the last line of a cell is displayed automatically. Use `print(df)` if you want the same output in a regular Python script.

## 2. Loading data

Real projects usually load data from files with functions like `pd.read_csv`, `pd.read_excel`, and `pd.read_parquet`. For this lesson, we read CSV text from memory, so the example runs as-is:

In [ ]:
from io import StringIO

csv_text = """claim_id,patient_id,date,provider,type,amount,paid
1,P001,2025-01-04,Clinic A,outpatient,120.50,True
2,P002,2025-01-09,Hospital B,inpatient,8450.00,True
3,P001,2025-02-11,Clinic A,outpatient,45.00,False
4,P003,2025-02-20,Clinic C,pharmacy,15.25,True
5,P002,2025-03-02,Hospital B,outpatient,310.00,
6,P004,2025-03-15,Clinic A,outpatient,,True
7,P003,2025-03-28,Hospital B,inpatient,2200.75,True
8,P005,2025-04-01,Clinic C,pharmacy,980.00,True
"""

claims = pd.read_csv(StringIO(csv_text))
claims

Notice that `paid` has a missing value, and `amount` has one too. Pandas marks those automatically as `NaN` (not a number) or `<NA>`. We'll handle them later.

To load a real file, you'd write `pd.read_csv("data/claims.csv")`.

Saving a DataFrame works the same way: `claims.to_csv("out.csv", index=False)`.

## 3. Inspecting data

Before you analyze anything, look at the data. These are the usual first checks:

In [ ]:
print("shape:", claims.shape)          # (rows, columns)
print()
print(claims.dtypes)
print()
print(claims.head(3))                  # first 3 rows
print()
print(claims.describe())               # summary statistics for numeric columns

`dtypes` shows what pandas thinks each column is. If a number column shows as `object` or the wrong type, check the data before you go further.

`info()` gives a quick overview, including how many values are missing:

In [ ]:
claims.info()

## 4. Selecting columns and rows

Getting a column returns a Series:

In [ ]:
claims["amount"]          # one column -> Series

Getting several columns returns a DataFrame. Note the double brackets:

In [ ]:
claims[["patient_id", "amount"]]

For rows, use **`loc`** (by label) and **`iloc`** (by position). The rule to remember: `loc` is for names and labels, and `iloc` is for numbers.

In [ ]:
print(claims.iloc[0])                     # first row, by position
print()
print(claims.loc[2, ["provider", "amount"]])  # row labeled 2, two columns
print()
print(claims.iloc[1:4, 2:5])             # rows 1-3, columns 2-4 (stop is excluded)

With the default index, `loc` and `iloc` often look the same, but they behave differently once you set a custom index. Keep that in mind.

## 5. Filtering

Filters work like NumPy masks. Combine conditions with `&` and `|`, and wrap each one in parentheses:

In [ ]:
large = claims[claims["amount"] > 500]
large

In [ ]:
clinic_outpatient = claims[
    (claims["provider"] == "Clinic A") & (claims["type"] == "outpatient")
]
clinic_outpatient

Other useful filter methods:

In [ ]:
print(claims[claims["type"].isin(["inpatient", "pharmacy"])])
print()
print(claims[claims["provider"].str.contains("Hospital")])

## 6. Creating and changing columns

Assigning to a new column name creates it. The operation works on the whole column at once, with no loop:

In [ ]:
claims["amount_with_tax"] = claims["amount"] * 1.07
claims[["amount", "amount_with_tax"]]

Use `np.where` for conditional values, or `.loc` to set values under a condition:

In [ ]:
claims["size"] = np.where(claims["amount"] > 500, "large", "small")

claims.loc[claims["paid"] == False, "size"] = "unpaid"

claims[["amount", "paid", "size"]]

Note the `== False` comparison above. A missing value in `paid` will not match either test, so it keeps its previous value. We'll handle that properly in the next section.

To remove a column:

In [ ]:
claims = claims.drop(columns=["amount_with_tax"])
claims.columns.tolist()

## 7. Missing values

Missing data shows up in real datasets all the time. Pandas gives you three main tools: find them, drop them, or fill them.

In [ ]:
print(claims.isna())                   # True where a value is missing
print()
print(claims.isna().sum())             # count of missing values per column

In [ ]:
print("rows with any missing value:")
print(claims[claims.isna().any(axis=1)])

**Dropping** removes rows with missing values. Use it with care, because you can lose a lot of data:

In [ ]:
claims.dropna(subset=["amount"])

**Filling** replaces missing values with something sensible, such as a default or a statistic:

In [ ]:
amount_filled = claims["amount"].fillna(claims["amount"].median())
print(amount_filled)

The `paid` column holds `True`, `False`, and missing values, so pandas stores it as `object`. Filling the gaps first, then converting to a real boolean type, makes later comparisons predictable:

In [ ]:
claims["paid"] = claims["paid"].fillna(False).astype(bool)
claims[["claim_id", "paid"]]

## 8. Sorting and ranking

In [ ]:
claims.sort_values("amount", ascending=False).head(3)

In [ ]:
claims.sort_values(["provider", "amount"])[["provider", "amount"]]

In [ ]:
claims["amount"].rank(ascending=False)

## 9. Grouping and aggregating

`groupby` splits the data into groups, applies a calculation to each group, and combines the results. It's the pandas version of a SQL `GROUP BY` or a spreadsheet pivot.

In [ ]:
claims.groupby("provider")["amount"].sum()

Use `.agg` to apply several summaries at once, and name the results:

In [ ]:
summary = claims.groupby("type").agg(
    total_claims=("claim_id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
)
summary

Groups can use more than one column. Here's the count and total by provider and type:

In [ ]:
claims.groupby(["provider", "type"])["amount"].agg(["count", "sum"])

`value_counts` is a quick way to count how often each value appears:

In [ ]:
claims["provider"].value_counts()

`pivot_table` reshapes a grouped result into a grid:

In [ ]:
pd.pivot_table(
    claims,
    index="provider",
    columns="type",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)

## 10. Combining tables

Real projects often keep data in several tables. Suppose we have a table of patients:

In [ ]:
patients = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003", "P004", "P005"],
    "plan": ["Gold", "Silver", "Gold", "Bronze", "Silver"],
    "age": [34, 58, 41, 27, 66],
})
patients

**`merge`** joins two tables on a shared column. This works like a SQL `JOIN`. The `how` argument controls which rows are kept:

- `inner`: only rows present in both tables
- `left`: all rows from the left table, matched where possible

In [ ]:
joined = claims.merge(patients, on="patient_id", how="left")
joined[["claim_id", "patient_id", "plan", "age", "amount"]]

**`concat`** stacks tables on top of each other, or side by side:

In [ ]:
jan = pd.DataFrame({"claim_id": [101], "amount": [50.0]})
feb = pd.DataFrame({"claim_id": [102], "amount": [75.0]})

pd.concat([jan, feb], ignore_index=True)

## 11. Dates

Once a column holds real dates, you can extract parts of them and do time-based analysis. Convert the text column first:

In [ ]:
claims["date"] = pd.to_datetime(claims["date"])

claims["month"] = claims["date"].dt.to_period("M")
claims["weekday"] = claims["date"].dt.day_name()

claims[["date", "month", "weekday"]].head(3)

In [ ]:
claims.groupby("month")["amount"].sum()

## 12. Chaining operations

You can write several steps in one expression. Each step returns a new DataFrame, so the chain reads from top to bottom. Wrapping it in parentheses lets you split it across lines:

In [ ]:
result = (
    claims
    .dropna(subset=["amount"])
    .query("paid == True")
    .groupby("provider")["amount"]
    .agg(["count", "mean"])
    .round(2)
    .sort_values("mean", ascending=False)
)
result

Chaining is often easier to read than a series of intermediate variables, and it's the style most data code uses.

## 13. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** How many claims are there for each patient? Sort the result from most to fewest claims.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
claims["patient_id"].value_counts()
```

</details>

**Exercise 2.** Find the average amount for paid claims only, by claim type. Ignore rows with a missing amount.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
(
    claims.dropna(subset=["amount"])
    .query("paid == True")
    .groupby("type")["amount"]
    .mean()
)
```

</details>

**Exercise 3.** Add the patient's plan to each claim, then find the total paid amount for each plan.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
merged = claims.merge(patients, on="patient_id", how="left")
merged[merged["paid"]].groupby("plan")["amount"].sum()
```

</details>

**Exercise 4.** Create a new column `is_large` that is `True` when the amount is over 1,000, and `False` otherwise. Treat missing amounts as `False`.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
claims["is_large"] = (claims["amount"] > 1000).fillna(False)
claims[["amount", "is_large"]]
```

</details>

## Summary

- **Series** are single columns. **DataFrames** are tables made of Series.
- **Load** with `read_csv` and other readers, then **inspect** with `head`, `info`, `describe`, and `isna`.
- Select with `[]` for columns, and `loc` and `iloc` for rows.
- Filter with boolean conditions, combining them with `&`, `|`, and parentheses.
- Create columns by assigning to them, and work on whole columns at once.
- Handle missing values with `isna`, `dropna`, and `fillna`.
- **`groupby`** plus `agg` covers most summary work.
- **`merge`** joins tables, and **`concat`** stacks them.
- Chain steps in parentheses to keep long workflows readable.